##### **IMDB API  Delta table**

In [ ]:
# Import required libraries
import requests
import pandas as pd
from pyspark.sql import SparkSession

# OMDb API configuration
api_key = os.environ.get("OMDB_API_KEY", "YOUR_OMDB_API_KEY")
movie_titles = ["Inception", "The Matrix", "The Dark Knight", "Interstellar", "Avengers: Endgame"]

# Collect movie data
movie_data = []
for title in movie_titles:
    url = f"http://www.omdbapi.com/?t={title}&apikey={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        movie_data.append(data)
    else:
        print(f"Failed to fetch data for {title}")

# Convert to DataFrame
df = pd.DataFrame(movie_data)

# Convert pandas to Spark
spark_df = SparkSession.builder.getOrCreate().createDataFrame(df)

# Save to Lakehouse Bronze as Delta Table
spark_df.write.mode("overwrite").format("delta").saveAsTable("bronze_omdb_api_movies")

# Optional: Show some data
display(spark_df.select("Title", "Year", "imdbRating", "Genre"))


In [ ]:
df_movies = spark.read.table("silver_movies_cleaned")
df_sample_ids = df_movies.select("ID").dropna().limit(200)

# Convert to Pandas for API looping
imdb_ids = [row["ID"] for row in df_sample_ids.collect()]

import requests
import pandas as pd
import time

OMDB_API_KEY = os.environ.get("OMDB_API_KEY", "YOUR_OMDB_API_KEY")OMDB_API_URL = "http://www.omdbapi.com/"

fetched_data = []

for imdb_id in imdb_ids:
    params = {"apikey": OMDB_API_KEY, "i": imdb_id}
    response = requests.get(OMDB_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        if data.get("Response") == "True":
            fetched_data.append({
                "imdb_id": imdb_id,
                "director": data.get("Director"),
            })
        else:
            print(f"Movie not found for {imdb_id}")
    else:
        print(f"Error fetching {imdb_id}: {response.status_code}")
    
    time.sleep(0.2)  # Be nice to the API

# Convert to DataFrame
omdb_df = pd.DataFrame(fetched_data)



In [ ]:
# Convert to Spark and save as bronze
spark_omdb_df = spark.createDataFrame(omdb_df)
spark_omdb_df.write.mode("overwrite").format("delta").saveAsTable("bronze_omdb_api_ratings")

display(spark_omdb_df)


In [ ]:
import pandas as pd
# Load data into pandas DataFrame from "/lakehouse/default/Files/movie_metadata.csv"
df = pd.read_csv("/lakehouse/default/Files/movie_metadata.csv")
display(df)

##### **Simulated SQL-like Table**

In [ ]:
from pyspark.sql.functions import col
import random
from datetime import datetime, timedelta
import pandas as pd

# Step 1: Load all movies from silver layer
df_movies = spark.read.table("silver_movies_cleaned").select("ID", "Title")
movie_list = df_movies.toPandas()

# Step 2: Generate synthetic reviews (1–3 per movie)
user_ids = [f"user_{i}" for i in range(1, 101)]
ratings_data = []

for index, row in movie_list.iterrows():
    imdb_id = row["ID"]
    title = row["Title"]
    num_reviews = random.randint(1, 3)  # AT MOST 3 reviews per movie

    for _ in range(num_reviews):
        user = random.choice(user_ids)
        rating = random.randint(1, 10)
        days_ago = random.randint(0, 1000)
        timestamp = (datetime.now() - timedelta(days=days_ago)).isoformat()

        ratings_data.append({
            "user_id": user,
            "imdb_id": imdb_id,
            "movie_title": title,
            "rating": rating,
            "timestamp": timestamp
        })

# Step 3: Write to Lakehouse as bronze_user_ratings
ratings_df = pd.DataFrame(ratings_data)
spark_df_ratings = spark.createDataFrame(ratings_df)
spark_df_ratings.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("bronze_user_ratings")

# Display result
display(spark_df_ratings)

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd

# Simulate users and movie titles
user_ids = [f"user_{i}" for i in range(1, 101)]
movie_titles = ["Inception", "The Matrix", "The Dark Knight", "Interstellar", "Avengers: Endgame"]

# Generate 500 synthetic ratings
ratings_data = []
for _ in range(500):
    user = random.choice(user_ids)
    movie = random.choice(movie_titles)
    rating = random.randint(1, 10)
    days_ago = random.randint(0, 1000)
    timestamp = (datetime.now() - timedelta(days=days_ago)).isoformat()

    ratings_data.append({
        "user_id": user,
        "movie_title": movie,
        "rating": rating,
        "timestamp": timestamp
    })

# Convert to DataFrame
ratings_df = pd.DataFrame(ratings_data)

# Convert to Spark and write to Lakehouse
spark_df_ratings = spark.createDataFrame(ratings_df)
spark_df_ratings.write.mode("overwrite").format("delta").saveAsTable("bronze_user_ratings")

display(spark_df_ratings)


### Loading CSV to Delta table

In [ ]:
df_movies = spark.read.format("csv").option("header","true").load("Files/movie_metadata.csv")
# df now is a Spark DataFrame containing CSV data from "Files/movie_metadata.csv".
display(df_movies)
df_movies.write.mode("overwrite").format("delta").saveAsTable("bronze_movies_metadata")


#### Creating Silver Movies Cleaned

In [ ]:
from pyspark.sql.functions import regexp_extract, col

df = spark.read.table("bronze_movies_metadata")

# Extract IMDb ID from the URL
silver_movies_df = df.withColumn(
    "ID",
    regexp_extract("movie_imdb_link", r"(tt\d+)", 1)
).select(
    col("ID"),
    col("movie_title").alias("Title"),
    col("title_year").alias("Year"),
    col("language").alias("Language"),
    col("budget").cast("int").alias("Budget"),
    col("gross").cast("int").alias("Revenue"),
    col("duration").cast("int").alias("Duration"),
    col("imdb_score").cast("double").alias("IMDB")
).dropna(subset=["ID", "Title", "Year"])

silver_movies_df.write.mode("overwrite").format("delta").saveAsTable("silver_movies_cleaned")


In [ ]:
df = spark.sql("SELECT * FROM MovieAnalyticsLakehouse.silver_movies_cleaned LIMIT 1000")
display(df)

#### Silver User ratings

In [ ]:
from pyspark.sql.functions import to_timestamp, col

# Step 1: Read from Bronze
df_bronze = spark.read.table("bronze_user_ratings")

# Step 2: Transform into Silver format
df_silver = df_bronze.select(
    col("user_id").cast("string"),
    col("imdb_id").cast("string"),
    col("rating").cast("double"),
    to_timestamp("timestamp").alias("rating_timestamp")
)

# Step 3: Write to Silver table
df_silver.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("silver_user_ratings")

# Optional: Show result
display(df_silver)


In [ ]:
spark.read.table("bronze_omdb_api_ratings").printSchema()


In [ ]:
df_omdb = spark.read.table("bronze_omdb_api_ratings")

silver_omdb = df_omdb.select(
    col("imdb_id"),
    col("director")
).dropna(subset=["imdb_id"])

silver_omdb.write.mode("overwrite").format("delta").saveAsTable("silver_omdb_api")


In [ ]:
df = spark.sql("SELECT * FROM MovieAnalyticsLakehouse.silver_movies_cleaned LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM MovieAnalyticsLakehouse.bronze_omdb_api_movies LIMIT 1000")
display(df)

In [ ]:
spark.read.table("silver_movies_cleaned").printSchema()


In [ ]:
movies_dim = spark.sql("""
SELECT 
    ID AS movie_key,
    Title AS title,
    Year AS release_year,
    Language AS language,
    Duration AS duration,
    Budget AS budget,
    Revenue AS revenue,
    IMDB AS imdb_rating
FROM silver_movies_cleaned
""")

movies_dim.write.mode("overwrite").format("delta").saveAsTable("gold_movies_dim")


In [ ]:
df = spark.sql("SELECT * FROM MovieAnalyticsLakehouse.silver_user_ratings")
display(df)

### `dim_movie` with CSD 2


In [ ]:
from pyspark.sql.functions import current_timestamp, lit, monotonically_increasing_id, col

# Step 1: Load both dataframes
df_movies = spark.read.table("silver_movies_cleaned")
df_omdb = spark.read.table("bronze_omdb_api_ratings")

# Step 2: Join on IMDb ID
df_joined = df_movies.join(
    df_omdb,
    df_movies["ID"] == df_omdb["imdb_id"],
    how="left"
)

# Step 3: Add SCD2 metadata
df_dim_movie = df_joined.withColumn("movie_sk", monotonically_increasing_id()) \
    .withColumn("start_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

# Step 4: Write to delta table
df_dim_movie.write.mode("overwrite").format("delta").saveAsTable("dim_movie")

# Optional: Display a preview
display(df_dim_movie.select("movie_sk", "Title", "director", "start_date", "is_current"))


### `dim_user` with CSD 2


In [ ]:
from pyspark.sql.functions import col

df_ratings = spark.read.table("silver_user_ratings")
df_users = df_ratings.select("user_id").distinct()

from pyspark.sql.functions import monotonically_increasing_id, expr
import random

# Generate random countries (or use a fixed list)
countries = ["USA", "UK", "India", "Canada", "Germany"]

# Register as a temporary view to use SQL to add synthetic data
df_users.createOrReplaceTempView("temp_users")

# Random registration date in the past ~3 years
df_users_meta = spark.sql(f"""
SELECT 
  monotonically_increasing_id() AS user_sk,
  user_id,
  date_sub(current_date(), CAST(rand() * 1000 AS INT)) AS registration_date,
  '{random.choice(countries)}' AS country
FROM temp_users
""")


In [ ]:
df_users_meta.write.mode("overwrite").format("delta").saveAsTable("dim_user")

# Optional preview
display(df_users_meta)


In [ ]:
tables = [
    "bronze_user_ratings",
    "bronze_omdb_api_ratings",
    "silver_movies_cleaned",
    "silver_user_ratings",
    "dim_user",
    "dim_movie",
    "fact_movie_ratings"
]

for table in tables:
    print(f"--- {table} ---")
    try:
        df = spark.read.table(table)
        df.printSchema()
        df.show(3, truncate=False)
    except Exception as e:
        print(f"Could not load {table}: {e}")
    print("\n\n")


In [ ]:
# When selecting or transforming:
df_movies = df_movies.withColumnRenamed("ID", "imdb_id")

In [ ]:
from pyspark.sql.functions import col

# Load data
df_ratings = spark.read.table("silver_user_ratings")
df_users = spark.read.table("dim_user").select("user_id", "user_sk")
df_movies = spark.read.table("dim_movie").select("imdb_id", "movie_sk")

# Join
df_fact = df_ratings.join(df_users, on="user_id", how="inner") \
                    .join(df_movies, on="imdb_id", how="inner")

# Final selection
fact_movie_ratings = df_fact.select(
    "user_sk", "movie_sk", "rating", col("rating_timestamp").alias("rating_time")
)

# Save
fact_movie_ratings.write.mode("overwrite").format("delta").saveAsTable("fact_movie_ratings")


In [ ]:
from pyspark.sql.functions import col

# Read current dim_movie
dim_movie = spark.read.table("dim_movie")

# Drop redundant ID column
dim_movie_cleaned = dim_movie.drop("ID")

# Overwrite the table
dim_movie_cleaned.write.mode("overwrite").format("delta").saveAsTable("dim_movie")




In [ ]:
spark.read.table("silver_user_ratings").printSchema()


In [ ]:
from pyspark.sql import functions as F

# Charger la table silver_user_ratings
df_ratings = spark.read.table("silver_user_ratings")

# Agréger par imdb_id
gold_rating_stats = df_ratings.groupBy("imdb_id") \
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("rating").alias("rating_count")
    )

# Sauvegarder dans la couche Gold
gold_rating_stats.write.mode("overwrite").format("delta").saveAsTable("gold_movie_rating_stats")

# Afficher un aperçu
display(gold_rating_stats)


In [ ]:
from pyspark.sql import functions as F

# Charger les stats depuis la table précédente
df_stats = spark.read.table("gold_movie_rating_stats")

# Filtrer les films avec au moins 20 votes et trier par note moyenne
gold_top_movies = df_stats \
    .filter(F.col("rating_count") >= 20) \
    .orderBy(F.col("avg_rating").desc()) \
    .limit(50)

# Écrire dans la couche Gold
gold_top_movies.write.mode("overwrite").format("delta").saveAsTable("gold_top_movies")

# Aperçu
display(gold_top_movies)


In [ ]:
from pyspark.sql import functions as F

# Charger les notations
df_ratings = spark.read.table("silver_user_ratings")

# Agréger par utilisateur
gold_user_activity = df_ratings.groupBy("user_id").agg(
    F.count("*").alias("total_ratings"),
    F.max("rating_timestamp").alias("last_activity")
)

# Sauvegarder en Gold
gold_user_activity.write.mode("overwrite").format("delta").saveAsTable("gold_user_activity")

# Aperçu
display(gold_user_activity)


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

# Lire les users distincts
df_ratings = spark.read.table("silver_user_ratings")
distinct_users = df_ratings.select("user_id").distinct()

# Ajouter une clé surrogate
dim_user = distinct_users.withColumn("user_sk", monotonically_increasing_id())

# Sauvegarde
dim_user.write.mode("overwrite").format("delta").saveAsTable("dim_user")
display(dim_user)

In [ ]:
# Lire les films distincts
df_movies = spark.read.table("silver_movies_cleaned").select("ID").dropna().distinct()

# Ajouter une clé technique
dim_movie = df_movies.withColumn("movie_sk", monotonically_increasing_id())

# Sauvegarde
dim_movie.write.mode("overwrite").format("delta").saveAsTable("dim_movie")
display(dim_movie)


In [ ]:
spark.read.table("fact_movie_ratings").show()


In [ ]:
print("Total lignes dans silver_user_ratings :", df_ratings.count())
print("Total lignes après join avec dim_user :", df_ratings.join(df_users, "user_id").count())
print("Total lignes après join complet :", df_fact.count())


In [ ]:
spark.read.table("silver_user_ratings").select("imdb_id").distinct().show(10)
spark.read.table("dim_movie").select("ID").distinct().show(10)

In [ ]:
from pyspark.sql.functions import col

# Lire les tables
df_ratings = spark.read.table("silver_user_ratings")
df_users = spark.read.table("dim_user").select("user_id", "user_sk")
df_movies = spark.read.table("dim_movie").select(col("ID").alias("imdb_id"), "movie_sk")

# Join avec les dimensions
df_fact = df_ratings \
    .join(df_users, on="user_id", how="inner") \
    .join(df_movies, on="imdb_id", how="inner")

# Sélectionner les colonnes finales
fact_movie_ratings = df_fact.select(
    "user_sk",
    "movie_sk",
    col("rating").cast("double").alias("rating"),
    col("rating_timestamp").alias("rating_time")
)

# Écrire
fact_movie_ratings.write.mode("overwrite").format("delta").saveAsTable("fact_movie_ratings")
display(fact_movie_ratings)
